In [19]:
import sys, platform
print(sys.executable)
print(platform.platform())

C:\Users\xxxAn\Desktop\NIH-crawl\.venv\Scripts\python.exe
Windows-10-10.0.19045-SP0


In [20]:
import pandas as pd
from pathlib import Path

In [21]:
from nih.paths import (
  ensure_dirs, 
  PROJECTS_ZIPS_DIR, 
  ABSTRACTS_ZIPS_DIR,
)

ensure_dirs()
print('Project zips:', PROJECTS_ZIPS_DIR)
print('Abstracts zips:', ABSTRACTS_ZIPS_DIR)

Project zips: D:\nih_data\raw\exporter_zips\projects
Abstracts zips: D:\nih_data\raw\exporter_zips\abstracts


In [ ]:
projects_zips = sorted(PROJECTS_ZIPS_DIR.glob('*.zip'))
abstracts_zips = sorted(ABSTRACTS_ZIPS_DIR.glob('*.zip'))

print('Projects ZIPs:', len(projects_zips))
print('Abstracts ZIPs:', len(abstracts_zips))
print('Example projects ZIP:', projects_zips[0] if projects_zips else None)
print('Example abstracts ZIP:', abstracts_zips[0] if abstracts_zips else None)

Projects ZIPs: 20
Abstracts ZIPs: 20
Example projects ZIP: D:\nih_data\raw\exporter_zips\projects\RePORTER_PRJ_C_FY2005.zip
Example abstracts ZIP: D:\nih_data\raw\exporter_zips\abstracts\RePORTER_PRJABS_C_FY2005.zip


In [23]:
proj_zip = projects_zips[-1]
abs_zip  = abstracts_zips[-1]
print(proj_zip.name, abs_zip.name)

RePORTER_PRJ_C_FY2024.zip RePORTER_PRJABS_C_FY2024.zip


In [ ]:
df_p = pd.read_csv(proj_zip, compression='zip', low_memory=False)
print(df_p.shape)
print(df_p.columns.tolist()[:10])
df_p.head(2)

(83314, 46)
['APPLICATION_ID', 'ACTIVITY', 'ADMINISTERING_IC', 'APPLICATION_TYPE', 'ARRA_FUNDED', 'AWARD_NOTICE_DATE', 'BUDGET_START', 'BUDGET_END', 'CFDA_CODE', 'CORE_PROJECT_NUM']


,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SERIAL_NUMBER,STUDY_SECTION,STUDY_SECTION_NAME,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,DIRECT_COST_AMT,INDIRECT_COST_AMT,TOTAL_COST,TOTAL_COST_SUB_PROJECT
0,9077077,I50,VA,5.0,N,2024-06-03,2015-10-01,2016-09-30,999.0,I50HX001238,...,1238,HCN1,HCN1 COIN REVIEW[HCN1],NaN,NaN,3.0,NaN,NaN,NaN,NaN
1,9662896,I50,VA,1.0,N,2024-05-24,2018-10-01,2019-09-30,999.0,I50HX002725,...,2725,HCN1,HCN1 COIN REVIEW[HCN1],NaN,NaN,1.0,NaN,NaN,NaN,NaN


In [26]:
df_a = pd.read_csv(abs_zip, compression='zip', low_memory=False)
print(df_a.shape)
print(df_a.columns.tolist()[:10])
df_a.head(2)

(77672, 2)
['APPLICATION_ID', 'ABSTRACT_TEXT']


,APPLICATION_ID,ABSTRACT_TEXT
0,11041785,The Targeted Clinical Research Program to Addr...
1,11041883,"To design, implement and manage ethical and st..."


In [ ]:
print('APPLICATION_ID in Projects?', 'APPLICATION_ID' in df_p.columns)
print('APPLICATION_ID in Abstracts?', 'APPLICATION_ID' in df_a.columns)

APPLICATION_ID in Projects? True
APPLICATION_ID in Abstracts? True


In [ ]:
df = df_p.merge(df_a[['APPLICATION_ID', 'ABSTRACT_TEXT']], on='APPLICATION_ID', how='left')

print('Joined shape:', df.shape)
missing = df['ABSTRACT_TEXT'].isna().mean()
print('Missing abstracts rate:', round(missing * 100, 2), '%')
df[['FY', 'PROJECT_TITLE', 'ABSTRACT_TEXT']].sample(3)

Joined shape: (83314, 47)
Missing abstracts rate: 6.87 %


,FY,PROJECT_TITLE,ABSTRACT_TEXT
58237,2024,Nano-CRISPR extracellular vesicle sensing syst...,Detection of reliable biomarkers that represen...
69197,2024,Return-to-work issues in healthcare workers di...,Project Summary/Abstract\nNurses are the large...
59271,2024,Planning and Evaluation Core,PLANNING AND EVALUATION CORE – ABSTRACT \nOur ...


In [ ]:
lengths = df['ABSTRACT_TEXT'].dropna().astype(str).str.len()
print('Abstract length (chars):', lengths.describe())

Abstract length (chars): count    77593.000000
mean      2711.372057
std        634.264184
min          1.000000
25%       2387.000000
50%       2877.000000
75%       3151.000000
max      23001.000000
Name: ABSTRACT_TEXT, dtype: float64


In [ ]:
COLS = [
    'APPLICATION_ID',
    'FY',
    'IC_NAME',
    'ACTIVITY',
    'PROJECT_TITLE',
    'ABSTRACT_TEXT',
]
toy_df = (
    df[COLS]
    .dropna(subset=['ABSTRACT_TEXT'])
    .sample(n=500, random_state=42)
    .reset_index(drop=True)
)
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

toy_path = DATA_DIR / 'toy_corpus_500.parquet'
toy_df.to_parquet(toy_path, index=False)
toy_df.to_parquet(toy_path, index=False)

print(f'Saved toy dataset to: {toy_path.resolve()}')
toy_df.head(3)


Saved toy dataset to: C:\Users\xxxAn\Desktop\NIH-crawl\data\toy_corpus_500.parquet


,APPLICATION_ID,FY,IC_NAME,ACTIVITY,PROJECT_TITLE,ABSTRACT_TEXT
0,10873210,2024,AGENCY FOR HEALTHCARE RESEARCH AND QUALITY,R01,Environmental hygiene strategies to decrease t...,PROJECT SUMMARY\n Infections due to antibiotic...
1,10765434,2024,NATIONAL INSTITUTE OF GENERAL MEDICAL SCIENCES,R35,Merging Signaling with Structure: Functions an...,PROJECT SUMMARY\nDespite not having a nervous ...
2,10941880,2024,NATIONAL INSTITUTE OF NURSING RESEARCH,R01,Improving Traumatic Stress in Black Women Expe...,PROJECT SUMMARY/ABSTRACT\n Homelessness and as...
